In [1]:
import pandas as pd
import numpy as np
from datetime import date, timedelta

In [2]:
df = pd.read_csv("Nat_Gas.csv")

df.columns = ["date", "price"]
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")
df.set_index("date", inplace=True)

df.head()

/var/folders/61/9_bd7j9j0rx2v0c23wplfm180000gn/T/ipykernel_13793/1125002441.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["date"])


,price
date,
2020-10-31,10.1
2020-11-30,10.3
2020-12-31,11.0
2021-01-31,10.9
2021-02-28,10.9


In [3]:
df["t"] = np.arange(len(df))
df["sin_12"] = np.sin(2 * np.pi * df["t"] / 12)
df["cos_12"] = np.cos(2 * np.pi * df["t"] / 12)

X = np.column_stack([
    np.ones(len(df)),
    df["t"],
    df["sin_12"],
    df["cos_12"]
])

y = df["price"].values
beta = np.linalg.lstsq(X, y, rcond=None)[0]

In [4]:
def estimate_gas_price(input_date):
    input_date = pd.to_datetime(input_date)

    months_since_start = (
        (input_date.year - df.index[0].year) * 12
        + (input_date.month - df.index[0].month)
    )

    sin_term = np.sin(2 * np.pi * months_since_start / 12)
    cos_term = np.cos(2 * np.pi * months_since_start / 12)

    X_new = np.array([
        1,
        months_since_start,
        sin_term,
        cos_term
    ])

    return float(X_new @ beta)

In [5]:
def price_storage_contract(
    injection_dates,
    withdrawal_dates,
    injection_volume,
    withdrawal_volume,
    max_storage,
    injection_rate,
    withdrawal_rate,
    storage_cost_per_day,
    price_function
):
    all_dates = sorted(set(injection_dates + withdrawal_dates))

    inventory = 0.0
    total_value = 0.0
    last_date = all_dates[0]

    for current_date in all_dates:
        days_passed = (current_date - last_date).days
        total_value -= inventory * storage_cost_per_day * days_passed

        if current_date in injection_dates:
            injected = min(injection_volume, injection_rate)
            if inventory + injected > max_storage:
                raise ValueError("Storage capacity exceeded")
            total_value -= injected * price_function(current_date)
            inventory += injected

        if current_date in withdrawal_dates:
            withdrawn = min(withdrawal_volume, withdrawal_rate)
            if withdrawn > inventory:
                raise ValueError("Insufficient inventory")
            total_value += withdrawn * price_function(current_date)
            inventory -= withdrawn

        last_date = current_date

    return total_value

In [6]:
injection_dates = [
    pd.Timestamp("2022-04-30"),
    pd.Timestamp("2022-05-31")
]

withdrawal_dates = [
    pd.Timestamp("2022-10-31"),
    pd.Timestamp("2022-11-30")
]

contract_value = price_storage_contract(
    injection_dates=injection_dates,
    withdrawal_dates=withdrawal_dates,
    injection_volume=1000,
    withdrawal_volume=1000,
    max_storage=2000,
    injection_rate=1000,
    withdrawal_rate=1000,
    storage_cost_per_day=0.01,
    price_function=estimate_gas_price
)

contract_value

-2573.8329613940496